# チュートリアル 01: ツールを持つエージェント - 実世界の機能を追加する

##  学習目標
このノートブックを完了すると、以下ができるようになります:
- ツールとは何か、なぜ重要なのかを理解する
- エージェントが呼び出せる Python 関数を作成する
- ツール定義のための型アノテーションと docstring を使用する
- 作成時および実行時にエージェントにツールを追加する
- エージェントがツールを使用するタイミングをどう決定するかを理解する

##  主要な概念

### ツールとは?
ツールは、エージェントが以下のために呼び出せる **Python 関数** です:
- リアルタイム情報の取得(天気、ニュース、価格)
- 計算の実行(通貨換算、距離)
- 外部 API へのアクセス(フライト、ホテル、レストラン)
- データベースやファイルとの対話

### 関数呼び出しのフロー
```
ユーザー: "パリの天気はどう?"
  ↓
エージェント: "天気ツールを呼び出す必要があります"
  ↓
ツール: get_weather("Paris") → "晴れ、22°C"
  ↓
エージェント: "パリの天気は晴れで、最高気温は 22°C です"
```

---

## ステップ 1: セットアップ

必要なものをインポートします。より良い型ヒントのために Pydantic も含めます。

In [ ]:
import asyncio
import os
from datetime import datetime
from random import randint, choice
from typing import Annotated

from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from agent_framework.azure import AzureOpenAIChatClient
from pydantic import Field
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    auth_kwargs = dict(credential=DefaultAzureCredential())
    print("🔐 認証方式: DefaultAzureCredential")

print("インポート成功！")

## ステップ 2: 最初のツールを作成する

シンプルな天気ツールを作成しましょう。重要な要素に注目してください:
1. **Docstring** - 関数が何をするかを説明(エージェントがこれを見ます!)
2. **型アノテーション** - パラメータの型を定義
3. **Field 説明** - エージェントにコンテキストを提供
4. **戻り値の型** - 関数が何を返すか

In [ ]:
def get_weather(
    location: Annotated[
        str,
        Field(
            description="天気を取得したい都市/場所（例: 'Paris', 'Tokyo', 'New York'）。"
        ),
    ],
) -> str:
    """
    指定した場所の現在の天気を取得します。
    気温（摂氏）と天候の概要を返します。
    """
    # 実際のアプリでは天気APIを呼び出します
    # ここでは簡易的にシミュレーションします
    conditions = ["晴れ", "晴れ時々くもり", "くもり", "雨", "嵐"]
    temp = randint(10, 35)
    condition = choice(conditions)

    return f"{location}の天気は{condition}、気温は{temp}℃です。"


# 関数を直接テスト
print(get_weather("Paris"))

## ステップ 3: ツールを持つエージェントを作成する

では、Travel Assistant に天気をチェックする機能を与えましょう!

In [ ]:
# 天気ツール付きでエージェントを作成
travel_agent = Agent(
    client=AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=AZURE_OPENAI_ENDPOINT,
        deployment_name=MODEL_DEPLOYMENT_NAME,
    ),
    name="TravelAssistant",
    instructions="""
    あなたは親切な旅行アシスタントです。目的地のおすすめ、旅行のコツ、
    そして現在の天気の確認ができます。

    ユーザーが天気について質問したら、get_weather ツールを使って正確な情報を提供してください。
    """,
    tools=[get_weather],  # ここにツールを追加！
)

print("天気ツール付きのエージェントを作成しました！")

## ステップ 4: ツールの使用をテストする

天気について質問して、エージェントがツールを使用する様子を見てみましょう。

In [ ]:
query = "バルセロナの天気はどんな感じですか？訪問を検討しています。"

print(f"ユーザー: {query}\n")
response = await travel_agent.run(query)
print(f"エージェント: {response}")

 **注意**: エージェントは自動的に `get_weather("Barcelona")` を呼び出し、その結果を応答に組み込みました!

## ステップ 5: さらにツールを追加する

旅行予算の計画を支援するために通貨コンバーターを追加しましょう。

In [ ]:
def convert_currency(
    amount: Annotated[float, Field(description="換算したい金額")],
    from_currency: Annotated[
        str, Field(description="元の通貨コード（例: 'USD', 'EUR', 'GBP'）")
    ],
    to_currency: Annotated[
        str, Field(description="換算先の通貨コード（例: 'USD', 'EUR', 'GBP'）")
    ],
) -> str:
    """
    ある通貨の金額を別の通貨に換算します。
    ざっくりした為替レートを使用します。
    """
    # 簡易的な為替レート（実際のアプリではリアルタイムAPIを使用）
    rates = {
        "USD": 1.0,
        "EUR": 0.92,
        "GBP": 0.79,
        "JPY": 149.50,
        "CAD": 1.36,
        "AUD": 1.53,
    }

    from_currency = from_currency.upper()
    to_currency = to_currency.upper()

    if from_currency not in rates or to_currency not in rates:
        return f"申し訳ありません。{from_currency} または {to_currency} の為替レートがありません。"

    # いったんUSDに換算してから、目的の通貨へ換算
    usd_amount = amount / rates[from_currency]
    converted = usd_amount * rates[to_currency]

    return f"{amount} {from_currency} は、おおよそ {converted:.2f} {to_currency} です"


# テスト
print(convert_currency(100, "USD", "EUR"))

## ステップ 6: フライト検索ツールを追加する

フライト検索をシミュレートしましょう(実際のアプリでは、Amadeus や Skyscanner のような API を呼び出します)。

In [ ]:
def search_flights(
    origin: Annotated[str, Field(description="出発地（都市名または空港コード）")],
    destination: Annotated[str, Field(description="目的地（都市名または空港コード）")],
    date: Annotated[str, Field(description="旅行日（YYYY-MM-DD形式）")],
) -> str:
    """
    指定した日付に、2地点間の利用可能なフライトを検索します。
    料金を含むフライト一覧の要約を返します。
    """
    # フライト検索結果をシミュレーション
    airlines = [
        "United",
        "Delta",
        "American",
        "Air France",
        "British Airways",
        "Lufthansa",
    ]

    flights = []
    for i in range(3):
        airline = choice(airlines)
        price = randint(200, 1200)
        duration = randint(2, 15)
        flights.append(f"- {airline}: ${price} ({duration}h {randint(0, 59)}m)")

    result = f"{date}に{origin}から{destination}へのフライトが{len(flights)}件見つかりました:\n"
    result += "\n".join(flights)
    return result


# テスト
print(search_flights("New York", "London", "2025-11-15"))

## ステップ 7: 強化された旅行エージェントを作成する

では、すべてのツールを持つエージェントを作成しましょう!

In [ ]:
# より高機能なエージェントを作成
enhanced_agent = Agent(
    client=AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=AZURE_OPENAI_ENDPOINT,
        deployment_name=MODEL_DEPLOYMENT_NAME,
    ),
    name="EnhancedTravelAssistant",
    instructions="""
    あなたはリアルタイムツールにアクセスできる、高度な旅行アシスタントです。

    できること:
    - 現在の天気を確認
    - 予算計画のための通貨換算
    - フライトの候補検索

    実用的な旅行情報を求められたら、適切なツールを使ってください。
    ツールの結果とあなたの知識を組み合わせ、包括的なアドバイスを提供してください。
    """,
    tools=[get_weather, convert_currency, search_flights],
)

print("3つのツール付きエージェントを作成しました！")

## ステップ 8: 複数ツールのクエリをテストする

複数のツールを必要とする質問をしてみましょう。

In [ ]:
# 天気と通貨換算が必要なクエリ
query = "東京旅行を計画しています。今の天気はどうですか？それと、500 USD は日本円でいくらですか？"

print(f"ユーザー: {query}\n")
response = await enhanced_agent.run(query)
print(f"エージェント: {response}")

In [ ]:
# すべてのツールが必要なクエリ
query = """
2025年12月1日にサンフランシスコからパリへ飛びたいです。
フライトの候補を見せて、現地の天気を教えて、1000 USD をユーロに換算してください。
"""

print(f"ユーザー: {query}\n")
response = await enhanced_agent.run(query)
print(f"エージェント: {response}")

##  ツール実行の理解

舞台裏で何が起こっているかを以下に示します:

1. **エージェントがクエリを分析** - どのツールが必要かを判断
2. **ツール呼び出しが行われる** - 抽出されたパラメータで関数が実行
3. **結果が収集される** - ツールの出力が集められる
4. **エージェントが応答を合成** - ツールの結果をコンテキストと組み合わせる

エージェントは以下を呼び出すことができます:
- **ツールなし** - クエリがそれらを必要としない場合
- **1つのツール** - シンプルなリクエストの場合
- **複数のツール** - 順次またはパラレルに
- **同じツールを複数回** - 必要な場合(例: 複数の都市の天気)

## ステップ 9: 実行時のツール

エージェントではなく、特定のクエリにツールを追加することもできます。

In [ ]:
# ツール無しのエージェントを作成
basic_agent = Agent(
    client=AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=AZURE_OPENAI_ENDPOINT,
        deployment_name=MODEL_DEPLOYMENT_NAME,
    ),
    name="BasicAgent",
    instructions="あなたは親切な旅行アシスタントです。",
    # ここではツールを定義しません！
)

# このクエリにだけツールを追加
query = "ローマの天気はどうですか？"
response = await basic_agent.run(query, tools=[get_weather])

print(f"ユーザー: {query}")
print(f"エージェント: {response}")

## ステップ 10: エラーハンドリングを持つツール

エラーを適切に処理する、より堅牢なツールを作成しましょう。

In [ ]:
def get_travel_advisory(
    country: Annotated[str, Field(description="渡航情報（注意喚起）を確認したい国")],
) -> str:
    """
    指定した国の最新の渡航勧告レベルを取得します。
    安全情報と推奨事項を返します。
    """
    try:
        # 注意喚起データをシミュレーション
        advisories = {
            "France": "レベル1: 通常の注意を払ってください",
            "Japan": "レベル1: 通常の注意を払ってください",
            "Mexico": "レベル2: 注意を強化してください",
            "Egypt": "レベル2: 注意を強化してください",
            "Afghanistan": "レベル4: 渡航しないでください",
        }

        country = country.strip().title()

        if country in advisories:
            return f"{country} の渡航情報: {advisories[country]}"
        else:
            return (
                f"{country} に関する具体的な渡航情報は見つかりませんでした。"
                "公式情報をご確認ください。"
            )

    except Exception as e:
        return f"渡航情報の取得中にエラーが発生しました: {str(e)}"


# テスト
print(get_travel_advisory("Japan"))
print(get_travel_advisory("Atlantis"))  # 架空の国

##  ツールのベストプラクティス

1. **明確な docstring** - エージェントはこれを使用して、いつツールを呼び出すかを理解します
2. **型アノテーション** - エージェントが正しいパラメータを抽出するのに役立ちます
3. **Field 説明** - 例とコンテキストを提供します
4. **エラーハンドリング** - クラッシュする代わりに有用なメッセージを返します
5. **一貫した戻り値** - 常に文字列を返します(または構造化出力を使用)
6. **説明的な名前** - 関数名は目的を明確に示すべきです

###  悪いツール
```python
def weather(x):
    return api.call(x)
```

###  良いツール
```python
def get_current_weather(
    location: Annotated[str, Field(description="City name or coordinates")],
) -> str:
    """Get real-time weather for a location."""
    try:
        # Implementation
        pass
    except Exception as e:
        return f"Error: {e}"
```

##  練習問題

これらのツールを作成してみてください:

1. **ホテル検索ツール** - 価格帯を指定して都市のホテルを検索
2. **距離計算機** - 2つの都市間の距離を計算
3. **レストラン検索** - 場所で料理の種類別にレストランを検索
4. **タイムゾーンコンバーター** - 異なるタイムゾーン間で時間を変換

In [ ]:
# 練習1: ホテル検索ツールを作成する
def search_hotels(
    city: Annotated[str, Field(description="ホテルを検索したい都市")],
    max_price: Annotated[int, Field(description="1泊あたりの上限金額（USD）")],
) -> str:
    """
    予算内で、指定した都市のホテルを検索します。
    """
    # ここに実装を書いてください
    pass


# ツールをテストしてみましょう！

##  重要なポイント

1. **ツールはエージェントの機能を拡張** し、LLM の知識だけでなく、それ以上のことができるようにします
2. **エージェントは賢く決定** し、ツールをいつどのように使用するかを判断します
3. **関数シグネチャが重要** - 良いアノテーション = より良いツール使用
4. **複数のツールが協力** して複雑なクエリを解決できます
5. **ツールは追加できる** - エージェント作成時またはクエリごとに

##  まだ足りないもの

私たちのエージェントはまだ:
-  以前の会話を覚えていません
-  クエリ間でコンテキストを維持できません
-  各 `run()` 呼び出しで新たに開始します

**次のチュートリアルでは**、ステートフルなマルチターン会話のための **Thread** を追加します!

---